# Experiment 7.2 — Two-layer multi-τ SNN training and persistence study

Aggregate-only notebook. Heavy training, per-seed diagnostics, probes, rasters, and checkpoint selection are performed by the Slurm workers and dependency finalizer. This notebook reads only method-level summary CSVs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists(): ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_2_two_layer_tau_training' / 'two_layer_tau_training_v1'
architectures = pd.read_csv(ART / 'architecture_table.csv')
performance = pd.read_csv(ART / 'performance_summary.csv')
dynamics = pd.read_csv(ART / 'dynamics_summary.csv')
deltas = pd.read_csv(ART / 'paired_deltas.csv')
calibration = pd.read_csv(ART / 'calibration_summary.csv')
ARCH_ORDER = ['234x234','34x234','34x34','34x345','34x45','4x4','5x5']
architectures

## Formal deployment comparison

`local_tsce` uses frozen L2 → Fixed250 → newly fitted LogisticRegression. `e2e_wc` uses output-LIF valid-length WholeCount. Error bars are mean ± SD across the three paired seeds.

In [ ]:
deploy = performance[(performance.split == 'test') & (((performance.training_family == 'local_tsce') & (performance.source == 'l2_fixed250')) | ((performance.training_family == 'e2e_wc') & (performance.source == 'e2e_wholecount')))].copy()
def deployment_plot(metric):
    fig, ax = plt.subplots(figsize=(12,5)); x = np.arange(len(ARCH_ORDER), dtype=float)
    combos = [('local_tsce','task_only'),('local_tsce','task_plus_reg'),('e2e_wc','task_only'),('e2e_wc','task_plus_reg')]
    for off,(family,reg) in zip(np.linspace(-.27,.27,len(combos)), combos):
        means=[]; stds=[]
        for arch in ARCH_ORDER:
            row=deploy[(deploy.architecture==arch)&(deploy.training_family==family)&(deploy.regularization==reg)]
            means.append(np.nan if row.empty else row.iloc[0][f'{metric}_mean']); stds.append(0 if row.empty else row.iloc[0][f'{metric}_std'])
        ax.errorbar(x+off,means,yerr=stds,marker='o',linestyle='none',label=f'{family}/{reg}')
    ax.set_xticks(x,ARCH_ORDER,rotation=30,ha='right'); ax.set_ylim(0,1); ax.set_ylabel(metric.replace('_',' ')); ax.legend(fontsize=8); fig.tight_layout(); return fig
deployment_plot('balanced_accuracy');

In [ ]:
deployment_plot('accuracy');

## E2E information-location probes

The E2E network is frozen before all probes. Compare L2 Fixed250, pre-output evidence Fixed250, post-output spike Fixed250, and formal WholeCount to locate where information is lost.

In [ ]:
p = performance[(performance.split=='test') & (performance.training_family=='e2e_wc') & performance.source.isin(['e2e_wholecount','l2_fixed250','pre_output_fixed250','post_output_fixed250']) & (performance.regularization=='task_only')]
fig,ax=plt.subplots(figsize=(12,5))
for source in ['e2e_wholecount','l2_fixed250','pre_output_fixed250','post_output_fixed250']:
    vals=[]
    for arch in ARCH_ORDER:
        row=p[(p.architecture==arch)&(p.source==source)]
        vals.append(np.nan if row.empty else row.iloc[0].balanced_accuracy_mean)
    ax.plot(ARCH_ORDER,vals,marker='o',label=source)
ax.set_ylim(0,1); ax.set_ylabel('test balanced accuracy'); ax.tick_params(axis='x',rotation=30); ax.legend(fontsize=8); fig.tight_layout();

## Paired deltas
The finalizer computes seed-paired differences before averaging; these are preferred over subtracting independently aggregated means.

In [ ]:
deltas[deltas.metric == 'balanced_accuracy']

## Shift-resolved hidden dynamics
These statistics are computed over the complete test set and resolved by hidden layer and shift group.

In [ ]:
cols=['architecture','training_family','regularization','layer','shift','n','valid_spikes_per_neuron_second_mean','p32_violation_fraction_mean','p84_violation_fraction_mean','mean_max_run_length_mean']
dynamics[[c for c in cols if c in dynamics.columns]].head(50)

In [ ]:
part=dynamics[(dynamics.training_family=='e2e_wc')&(dynamics.regularization=='task_only')&(dynamics.layer=='L2')]
fig,ax=plt.subplots(figsize=(12,5))
for shift in sorted(part['shift'].dropna().unique()):
    vals=[]
    for arch in ARCH_ORDER:
        row=part[(part.architecture==arch)&(part['shift']==shift)]
        vals.append(np.nan if row.empty else row.iloc[0].valid_spikes_per_neuron_second_mean)
    ax.plot(ARCH_ORDER,vals,marker='o',label=f'shift {int(shift)}')
ax.set_ylabel('L2 spikes / neuron / s'); ax.tick_params(axis='x',rotation=30); ax.legend(); fig.tight_layout();

## Regularizer calibration summary
Calibration is architecture- and training-family-specific because WholeCount and timestep CE produce different task gradients.

In [ ]:
calibration

## Interpretation checklist

- Main controlled path: `34x34 → 34x345 → 34x45` with L1 fixed at `(3,4)`.
- L2 probe high but E2E WholeCount low: hidden representation survives; streaming readout is the bottleneck.
- Pre-output probe high but post-output probe low: output LIF spike conversion is a bottleneck.
- E2E L2 probe lower than local L2+Linear: WholeCount end-to-end optimization damaged hidden representation.
- Regularization lowers persistence and restores BA: sustained firing is a causal contributor.
- Persistence falls without BA recovery: long τ likely also harms local information through temporal smoothing/mixing.